# Benin Insights 2025 - Dashboard Frontend (Prototype Colab)
## Pipeline GDELT Events + GKG - Visualisation interactive
**iSHEERO x DataCamp Donates - Role : Data Engineer**

> Lance toutes les cellules dans l'ordre : Runtime -> Run all

---

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'pandas', 'numpy'], check=True)
print('Packages prets')

Packages prets


## 1 - Generation des donnees prototype GDELT Events + GKG Benin

> **Tables simulees** : `gdeltv2.events` + `gdeltv2.gkg`
> **Periode** : 2025-01-01 a 2025-12-31

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N = 8000

# =============================================================================
# PARTIE 1 : TABLE EVENTS (gdeltv2.events)
# =============================================================================

event_codes = {
    '010': 'Declaration publique',
    '020': 'Appel action',
    '036': 'Critique politique',
    '040': 'Consultation diplomatique',
    '050': 'Appel cooperation',
    '060': 'Cooperation materielle',
    '070': 'Aide/assistance',
    '100': 'Demande',
    '110': 'Rejet/opposition',
    '120': 'Menace',
    '130': 'Protestation',
    '140': 'Refus cooperation',
    '145': 'Manifestation/revolte',
    '170': 'Coercition',
    '180': 'Attaque',
}
ek = list(event_codes.keys())
ew = np.array([.12,.08,.10,.09,.08,.07,.06,.07,.07,.04,.06,.04,.03,.03,.03])
ew /= ew.sum()
chosen_ev = np.random.choice(ek, N, p=ew)

cities_info = {
    'Cotonou':       {'lat':6.3654,  'lon':2.4183,  'zone':'Sud',    'dept':'Littoral',   'nord':False},
    'Porto-Novo':    {'lat':6.4969,  'lon':2.6289,  'zone':'Sud',    'dept':'Oueme',      'nord':False},
    'Parakou':       {'lat':9.3370,  'lon':2.6282,  'zone':'Nord',   'dept':'Borgou',     'nord':True},
    'Abomey-Calavi': {'lat':6.4484,  'lon':2.3552,  'zone':'Sud',    'dept':'Atlantique', 'nord':False},
    'Bohicon':       {'lat':7.1824,  'lon':2.0660,  'zone':'Centre', 'dept':'Zou',        'nord':False},
    'Natitingou':    {'lat':10.3081, 'lon':1.3784,  'zone':'Nord',   'dept':'Atacora',    'nord':True},
    'Kandi':         {'lat':11.1342, 'lon':2.9394,  'zone':'Nord',   'dept':'Alibori',    'nord':True},
    'Lokossa':       {'lat':6.6392,  'lon':1.7153,  'zone':'Sud',    'dept':'Mono',       'nord':False},
    'Ouidah':        {'lat':6.3600,  'lon':2.0852,  'zone':'Sud',    'dept':'Atlantique', 'nord':False},
    'Abomey':        {'lat':7.1856,  'lon':1.9861,  'zone':'Centre', 'dept':'Zou',        'nord':False},
    'Djougou':       {'lat':9.7082,  'lon':1.6599,  'zone':'Nord',   'dept':'Donga',      'nord':True},
    'Malanville':    {'lat':11.8683, 'lon':3.3836,  'zone':'Nord',   'dept':'Alibori',    'nord':True},
}
city_names = list(cities_info.keys())
city_weights = np.array([.303,.151,.124,.100,.060,.050,.040,.040,.040,.035,.030,.027])
city_weights /= city_weights.sum()
chosen_cities = np.random.choice(city_names, N, p=city_weights)

actors = ['Gouvernement','Parti politique','ONG','Militaire','Medias',
          'Diplomate','Syndicat','Citoyen','Police','Institution internationale']
actor_w = np.array([.25,.18,.12,.08,.12,.07,.06,.05,.04,.03])
actor_w /= actor_w.sum()

sources = ['bbc.com','rfi.fr','beninwebtv.com','lanouvelletribune.bj',
           'matinlibre.com','24haubenin.com','fraternite.bj','apanews.net']
source_regions = {'bbc.com':'International','rfi.fr':'International',
                  'beninwebtv.com':'Benin','lanouvelletribune.bj':'Benin',
                  'matinlibre.com':'Benin','24haubenin.com':'Benin',
                  'fraternite.bj':'Benin','apanews.net':'Afrique'}

# --- PERIODE FIXE : 2025-01-01 -> 2025-12-31 ---
date_start = datetime(2025, 1, 1)
date_end   = datetime(2025, 12, 31)
n_days = (date_end - date_start).days + 1  # 365 jours
dates = [date_start + timedelta(days=int(d)) for d in np.random.choice(n_days, N)]

neg_events = ['110','120','130','140','145','170','180']
is_neg = np.isin(chosen_ev, neg_events)

goldstein = np.where(is_neg,
    np.random.normal(-3.5, 2.5, N),
    np.random.normal(2.0, 2.0, N)).clip(-10, 10)

avg_tone = np.where(is_neg,
    np.random.normal(-4.2, 3.0, N),
    np.random.normal(1.8, 2.5, N)).clip(-20, 20)

quad_class = np.where(chosen_ev == '060', 2,
             np.where(chosen_ev == '070', 2,
             np.where(np.isin(chosen_ev, ['145','170','180']), 4,
             np.where(is_neg, 3, 1))))

num_mentions = np.random.negative_binomial(3, 0.3, N) + 1
num_articles = np.random.negative_binomial(2, 0.4, N) + 1
chosen_sources = np.random.choice(sources, N)
chosen_actors  = np.random.choice(actors, N, p=actor_w)

df = pd.DataFrame({
    'SQLDATE':            [d.strftime('%Y%m%d') for d in dates],
    'date':               dates,
    'Actor1Name':         chosen_actors,
    'Actor1CountryCode':  np.where(np.random.rand(N)<0.7,'BN',
                            np.random.choice(['FR','US','NG','GH','TG'],N)),
    'EventCode':          chosen_ev,
    'EventLabel':         [event_codes[e] for e in chosen_ev],
    'QuadClass':          quad_class,
    'GoldsteinScale':     goldstein.round(2),
    'NumMentions':        num_mentions,
    'NumArticles':        num_articles,
    'AvgTone':            avg_tone.round(2),
    'ActionGeo_FullName': chosen_cities,
    'ActionGeo_Lat':      [cities_info[c]['lat'] for c in chosen_cities],
    'ActionGeo_Long':     [cities_info[c]['lon'] for c in chosen_cities],
    'SourceDomain':       chosen_sources,
})

# =============================================================================
# PARTIE 2 : TABLE GKG (gdeltv2.gkg) - enrichissement
# =============================================================================
# La table GKG (Global Knowledge Graph) ajoute des dimensions thematiques,
# entites nommees, organisations, et un tone decompose en 6 composantes.

# --- GKG Themes (V2Themes) ---
gkg_themes_pool = [
    'POLITICS', 'ELECTION', 'GOVERNMENT', 'DEMOCRACY',
    'SECURITY', 'MILITARY', 'TERRORISM', 'PROTEST',
    'ECONOMY', 'TRADE', 'AGRICULTURE', 'FINANCE',
    'HEALTH', 'EDUCATION', 'INFRASTRUCTURE',
    'CLIMATE', 'ENVIRONMENT', 'DROUGHT', 'FLOOD',
    'CORRUPTION', 'HUMAN_RIGHTS', 'MIGRATION', 'RELIGION',
    'DEVELOPMENT', 'AID', 'DIPLOMACY',
    'MEDIA', 'TECHNOLOGY', 'ENERGY', 'MINING',
    'WB_2862_FOOD_SECURITY', 'TAX_POLITICAL_PARTY',
    'UNGP_FORESTS_RIVERS', 'EPU_POLICY_UNCERTAINTY',
    'GENDER', 'YOUTH', 'POVERTY',
]

# Chaque evenement a 1 a 5 themes GKG
def pick_themes(event_code, is_negative):
    n_themes = np.random.randint(1, 6)
    # Biais : evenements negatifs -> themes securitaires
    if is_negative:
        weights = np.array([2 if t in ['SECURITY','MILITARY','TERRORISM','PROTEST',
                    'CORRUPTION','HUMAN_RIGHTS','MIGRATION'] else 1
                    for t in gkg_themes_pool], dtype=float)
    else:
        weights = np.array([2 if t in ['POLITICS','ECONOMY','DEVELOPMENT','AID',
                    'DIPLOMACY','TRADE','EDUCATION','HEALTH'] else 1
                    for t in gkg_themes_pool], dtype=float)
    weights /= weights.sum()
    selected = np.random.choice(gkg_themes_pool, n_themes, replace=False, p=weights)
    return ';'.join(selected)

df['GKG_Themes'] = [pick_themes(ec, neg) for ec, neg in zip(chosen_ev, is_neg)]
df['GKG_NumThemes'] = df['GKG_Themes'].apply(lambda x: len(x.split(';')))

# --- GKG Persons (V2Persons) ---
persons_pool = [
    'Patrice Talon', 'Mariam Chabi Talata', 'Joseph Djogbenou',
    'Abdoulaye Bio Tchane', 'Aurélien Agbenonci', 'Romuald Wadagni',
    'Alassane Séidou', 'Samou Séidou Adambi', 'Claudine Prudencio',
    'Louis Vlavonou', 'Razack Adeoti', 'Mohamed Bazoum',
    'Faure Gnassingbé', 'Bola Tinubu', 'Emmanuel Macron',
    'Antonio Guterres', 'Akinwumi Adesina', 'Moussa Faki',
    '', '', '', '',  # beaucoup d'evenements n'ont pas de personne
]

def pick_persons():
    n = np.random.choice([0,0,0,1,1,2,3])
    if n == 0:
        return ''
    selected = np.random.choice([p for p in persons_pool if p], min(n, 5), replace=False)
    return ';'.join(selected)

df['GKG_Persons'] = [pick_persons() for _ in range(N)]
df['GKG_NumPersons'] = df['GKG_Persons'].apply(lambda x: len([p for p in x.split(';') if p]))

# --- GKG Organizations (V2Organizations) ---
orgs_pool = [
    'Gouvernement du Benin', 'Assemblee Nationale', 'Cour Constitutionnelle',
    'UNDP', 'UNESCO', 'World Bank', 'IMF', 'African Union',
    'ECOWAS', 'EU', 'USAID',
    'FCBE', 'Union Progressiste', 'Bloc Republicain', 'Forces Cauris',
    'Benin Telecom', 'Port de Cotonou', 'SBEE', 'SONEB',
    'Croix-Rouge', 'Amnesty International', 'Transparency International',
    '', '', '',
]

def pick_orgs():
    n = np.random.choice([0,0,1,1,1,2,2,3])
    if n == 0:
        return ''
    selected = np.random.choice([o for o in orgs_pool if o], min(n, 4), replace=False)
    return ';'.join(selected)

df['GKG_Organizations'] = [pick_orgs() for _ in range(N)]
df['GKG_NumOrgs'] = df['GKG_Organizations'].apply(lambda x: len([o for o in x.split(';') if o]))

# --- GKG V2Tone decompose (6 composantes) ---
# Format GDELT V2Tone : tone, pos_score, neg_score, polarity, activity_ref_density, self_group_ref_density
df['GKG_V2Tone_Positive'] = np.clip(np.where(is_neg,
    np.random.exponential(1.5, N),
    np.random.exponential(4.0, N)), 0, 20).round(2)
df['GKG_V2Tone_Negative'] = np.clip(np.where(is_neg,
    np.random.exponential(5.0, N),
    np.random.exponential(1.8, N)), 0, 20).round(2)
df['GKG_V2Tone_Polarity'] = (df['GKG_V2Tone_Positive'] + df['GKG_V2Tone_Negative']).round(2)
df['GKG_V2Tone_ActivityDensity'] = np.clip(np.random.exponential(3.0, N), 0, 25).round(2)
df['GKG_V2Tone_SelfGroupDensity'] = np.clip(np.random.exponential(1.0, N), 0, 15).round(2)
df['GKG_V2Tone_WordCount'] = np.random.randint(50, 3000, N)

# --- GKG GCAM (Global Content Analysis Measures) - scores emotionnels ---
df['GCAM_Anger']     = np.clip(np.where(is_neg, np.random.exponential(2.5, N), np.random.exponential(0.5, N)), 0, 15).round(3)
df['GCAM_Fear']      = np.clip(np.where(is_neg, np.random.exponential(2.0, N), np.random.exponential(0.4, N)), 0, 15).round(3)
df['GCAM_Joy']       = np.clip(np.where(~is_neg, np.random.exponential(2.5, N), np.random.exponential(0.3, N)), 0, 15).round(3)
df['GCAM_Sadness']   = np.clip(np.where(is_neg, np.random.exponential(1.8, N), np.random.exponential(0.4, N)), 0, 15).round(3)
df['GCAM_Trust']     = np.clip(np.where(~is_neg, np.random.exponential(3.0, N), np.random.exponential(0.8, N)), 0, 15).round(3)
df['GCAM_Surprise']  = np.clip(np.random.exponential(1.0, N), 0, 10).round(3)

# Emotion dominante
gcam_cols = ['GCAM_Anger','GCAM_Fear','GCAM_Joy','GCAM_Sadness','GCAM_Trust','GCAM_Surprise']
df['GCAM_DominantEmotion'] = df[gcam_cols].idxmax(axis=1).str.replace('GCAM_','')

# --- GKG DocumentIdentifier (URL source) ---
url_templates = {
    'bbc.com': 'https://www.bbc.com/afrique/articles/benin-{slug}',
    'rfi.fr': 'https://www.rfi.fr/fr/afrique/{date}-benin-{slug}',
    'beninwebtv.com': 'https://beninwebtv.com/{date}/{slug}',
    'lanouvelletribune.bj': 'https://lanouvelletribune.bj/{slug}',
    'matinlibre.com': 'https://matinlibre.com/2025/{slug}',
    '24haubenin.com': 'https://24haubenin.com/?{slug}',
    'fraternite.bj': 'https://fraternite.bj/article/{slug}',
    'apanews.net': 'https://apanews.net/benin-{slug}/',
}
def gen_url(source, date_str):
    tpl = url_templates.get(source, 'https://{source}/{slug}')
    slug = f"evt-{np.random.randint(10000, 99999)}"
    return tpl.format(slug=slug, date=date_str, source=source)

df['GKG_DocumentIdentifier'] = [gen_url(s, d.strftime('%Y/%m/%d'))
                                  for s, d in zip(df['SourceDomain'], df['date'])]

# --- GKG SharingImage ---
df['GKG_HasImage'] = np.random.choice([0, 1], N, p=[0.35, 0.65])

# --- GKG TranslationInfo ---
lang_pool = ['fra','fra','fra','eng','fra','fra']
df['GKG_Language'] = np.random.choice(lang_pool, N)
df['GKG_IsTranslated'] = np.where(df['GKG_Language'] != 'fra', 1, 0)

# =============================================================================
# PARTIE 3 : FEATURE ENGINEERING (Events + GKG combines)
# =============================================================================

quad_labels = {1:'Cooperation verbale',2:'Cooperation materielle',
               3:'Conflit verbal',4:'Conflit materiel'}
df['QuadLabel']       = df['QuadClass'].map(quad_labels)
df['IsConflict']      = (df['QuadClass'] >= 3).astype(int)
df['IsNegative']      = (df['AvgTone'] < 0).astype(int)
df['MediaWeight']     = df['NumMentions'] * df['NumArticles']
df['IsNorthBenin']    = df['ActionGeo_FullName'].map(lambda c: int(cities_info[c]['nord']))
df['ZoneBenin']       = df['ActionGeo_FullName'].map(lambda c: cities_info[c]['zone'])
df['DepartementBenin']= df['ActionGeo_FullName'].map(lambda c: cities_info[c]['dept'])
df['SourceRegion']    = df['SourceDomain'].map(source_regions)
df['GoldsteinNorm']   = ((df['GoldsteinScale'] + 10) / 20).round(4)
df['ToneNorm']        = ((df['AvgTone'] + 20) / 40).round(4)
df['month']           = pd.to_datetime(df['date']).dt.to_period('M')

# Features croisees Events x GKG
df['GKG_ThemeConflict'] = df['GKG_Themes'].apply(
    lambda t: int(any(x in t for x in ['SECURITY','MILITARY','TERRORISM','PROTEST'])))
df['GKG_ThemeEconomy']  = df['GKG_Themes'].apply(
    lambda t: int(any(x in t for x in ['ECONOMY','TRADE','FINANCE','AGRICULTURE'])))
df['GKG_ThemeGovern']   = df['GKG_Themes'].apply(
    lambda t: int(any(x in t for x in ['POLITICS','ELECTION','GOVERNMENT','DEMOCRACY'])))
df['GKG_ThemeHumanDev'] = df['GKG_Themes'].apply(
    lambda t: int(any(x in t for x in ['HEALTH','EDUCATION','POVERTY','GENDER','YOUTH'])))
df['GKG_ThemeEnviro']   = df['GKG_Themes'].apply(
    lambda t: int(any(x in t for x in ['CLIMATE','ENVIRONMENT','DROUGHT','FLOOD'])))
df['GKG_ToneGap']       = (df['GKG_V2Tone_Positive'] - df['GKG_V2Tone_Negative']).round(2)
df['GKG_EmotionIntensity'] = df[gcam_cols].sum(axis=1).round(3)

def tone_cat(t):
    if t < -5: return 'Tres negatif'
    elif t < -2: return 'Negatif'
    elif t < 2: return 'Neutre'
    elif t < 5: return 'Positif'
    else: return 'Tres positif'

def gold_bin(g):
    if g < -7: return 'Tres destabilisant'
    elif g < -3: return 'Destabilisant'
    elif g < 0: return 'Leg. negatif'
    elif g < 3: return 'Neutre'
    elif g < 7: return 'Stabilisant'
    else: return 'Tres stabilisant'

df['ToneCategory'] = df['AvgTone'].apply(tone_cat)
df['GoldsteinBin'] = df['GoldsteinScale'].apply(gold_bin)

print(f'Dataset genere : {len(df):,} evenements, {df.shape[1]} colonnes')
print(f'Periode : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'Tables  : gdeltv2.events + gdeltv2.gkg (fusionnees)')
print(f'Taux conflit : {df["IsConflict"].mean()*100:.1f}%')
print(f'AvgTone moy  : {df["AvgTone"].mean():.3f}')
print(f'Goldstein moy: {df["GoldsteinScale"].mean():.3f}')
print(f'GKG Themes uniques   : {len(gkg_themes_pool)}')
print(f'GKG Themes/evt (moy) : {df["GKG_NumThemes"].mean():.1f}')
print(f'GKG Persons detectees: {(df["GKG_NumPersons"]>0).sum():,}')
print(f'GKG Orgs detectees   : {(df["GKG_NumOrgs"]>0).sum():,}')
print(f'GCAM Emotions        : 6 dimensions')
df.head(3)


Dataset genere : 8,000 evenements, 28 colonnes
Periode : 2025-05-03 -> 2026-05-02
Taux conflit : 30.0%
AvgTone moy  : 0.007
Goldstein moy: 0.347


,SQLDATE,date,Actor1Name,Actor1CountryCode,EventCode,EventLabel,QuadClass,GoldsteinScale,NumMentions,NumArticles,...,MediaWeight,IsNorthBenin,ZoneBenin,DepartementBenin,SourceRegion,GoldsteinNorm,ToneNorm,month,ToneCategory,GoldsteinBin
0,20251026,2025-10-26 09:44:12.337439,Parti politique,NG,040,Consultation diplomatique,1,1.85,4,3,...,12,0,Centre,Zou,Benin,0.5925,0.4995,2025-10,Neutre,Neutre
1,20260321,2026-03-21 09:44:12.337439,Diplomate,BN,170,Coercition,4,-5.68,4,5,...,20,0,Centre,Zou,Benin,0.2160,0.3492,2026-03,Tres negatif,Destabilisant
2,20260309,2026-03-09 09:44:12.337439,Gouvernement,BN,110,Rejet/opposition,3,-5.14,2,4,...,8,0,Sud,Littoral,Benin,0.2430,0.3688,2026-03,Tres negatif,Destabilisant


## 2 - Rapport pipeline

In [ ]:
print('='*60)
print('  RAPPORT PIPELINE GDELT BENIN (BN) - Events + GKG')
print('='*60)
print(f'\nSOURCES')
print(f'  Table Events : gdeltv2.events')
print(f'  Table GKG    : gdeltv2.gkg')
print(f'  Jointure     : SQLDATE + SourceDomain')
print(f'\nVOLUME')
print(f'  Total     : {len(df):,}')
print(f'  Features  : {df.shape[1]}')
print(f'  Periode   : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'\nINDICATEURS EVENTS')
print(f'  AvgTone   : {df["AvgTone"].mean():.3f}')
print(f'  Goldstein : {df["GoldsteinScale"].mean():.3f}')
print(f'  Conflit   : {df["IsConflict"].mean()*100:.1f}%')
print(f'  Negatif   : {df["IsNegative"].mean()*100:.1f}%')
print(f'\nINDICATEURS GKG')
print(f'  Themes/evt (moy)  : {df["GKG_NumThemes"].mean():.1f}')
print(f'  V2Tone Positif    : {df["GKG_V2Tone_Positive"].mean():.2f}')
print(f'  V2Tone Negatif    : {df["GKG_V2Tone_Negative"].mean():.2f}')
print(f'  ToneGap (pos-neg) : {df["GKG_ToneGap"].mean():.2f}')
print(f'  WordCount moyen   : {df["GKG_V2Tone_WordCount"].mean():.0f}')
print(f'\nGCAM EMOTIONS')
for emo in ['GCAM_Anger','GCAM_Fear','GCAM_Joy','GCAM_Sadness','GCAM_Trust','GCAM_Surprise']:
    print(f'  {emo:<20}: {df[emo].mean():.3f}')
print(f'  Dominante : {df["GCAM_DominantEmotion"].value_counts().index[0]}')
print(f'\nTOP 5 THEMES GKG')
from collections import Counter
all_themes = ';'.join(df['GKG_Themes']).split(';')
for theme, cnt in Counter(all_themes).most_common(5):
    print(f'  {theme:<30} {cnt:>5}')
print(f'\nTOP 5 VILLES')
for city, cnt in df['ActionGeo_FullName'].value_counts().head(5).items():
    print(f'  {city:<20} {cnt:>5}')
print(f'\nTOP 5 PERSONNES GKG')
all_persons = [p for p in ';'.join(df['GKG_Persons']).split(';') if p]
for pers, cnt in Counter(all_persons).most_common(5):
    print(f'  {pers:<30} {cnt:>5}')
print(f'\nTOP 5 ORGANISATIONS GKG')
all_orgs = [o for o in ';'.join(df['GKG_Organizations']).split(';') if o]
for org, cnt in Counter(all_orgs).most_common(5):
    print(f'  {org:<30} {cnt:>5}')
print(f'\nQUADCLASS')
for lbl, cnt in df['QuadLabel'].value_counts().items():
    print(f'  {lbl:<30} {cnt:>5} ({cnt/len(df)*100:.1f}%)')
print('='*60)


  RAPPORT PIPELINE GDELT BENIN (BN)

VOLUME
  Total     : 8,000
  Features  : 28
  Periode   : 2025-05-03 -> 2026-05-02

INDICATEURS
  AvgTone   : 0.007
  Goldstein : 0.347
  Conflit   : 30.0%
  Negatif   : 44.5%

TOP 5 VILLES
  Cotonou               2441
  Porto-Novo            1231
  Parakou               1010
  Abomey-Calavi          749
  Bohicon                469

QUADCLASS
  Cooperation verbale             4513 (56.4%)
  Conflit verbal                  1671 (20.9%)
  Cooperation materielle          1083 (13.5%)
  Conflit materiel                 733 (9.2%)


## 3 - Dashboard interactif dans Colab

In [ ]:
from IPython.display import HTML, display
from collections import Counter

monthly = df.groupby('month').size().reset_index(name='count').tail(12)
months_labels = [str(m) for m in monthly['month'].tolist()]
months_vals   = monthly['count'].tolist()

tone_by_month = df.groupby('month')['AvgTone'].mean().reset_index().tail(12)
gold_by_month = df.groupby('month')['GoldsteinScale'].mean().reset_index().tail(12)
tone_vals = [round(float(v), 2) for v in tone_by_month['AvgTone'].tolist()]
gold_vals = [round(float(v), 2) for v in gold_by_month['GoldsteinScale'].tolist()]

conflict_by_month = df.groupby('month')['IsConflict'].mean().reset_index().tail(12)
conflict_vals = [round(float(v)*100, 1) for v in conflict_by_month['IsConflict'].tolist()]

quad_counts = df['QuadLabel'].value_counts()
quad_labels_list = quad_counts.index.tolist()
quad_vals_list   = [int(v) for v in quad_counts.values.tolist()]

top_cities = df['ActionGeo_FullName'].value_counts().head(8)
city_labels = top_cities.index.tolist()
city_counts = [int(v) for v in top_cities.values.tolist()]

tone_dist = df['ToneCategory'].value_counts()
tone_cats = tone_dist.index.tolist()
tone_cnts = [int(v) for v in tone_dist.values.tolist()]

# --- GKG : Top themes ---
all_themes = ';'.join(df['GKG_Themes']).split(';')
theme_counter = Counter(all_themes).most_common(10)
theme_labels = [t[0] for t in theme_counter]
theme_counts = [t[1] for t in theme_counter]

# --- GKG : GCAM emotions moyennes ---
gcam_labels = ['Anger','Fear','Joy','Sadness','Trust','Surprise']
gcam_means  = [round(float(df[f'GCAM_{e}'].mean()), 2) for e in gcam_labels]

# --- GKG : Emotion dominante distribution ---
emo_dist = df['GCAM_DominantEmotion'].value_counts()
emo_labels = emo_dist.index.tolist()
emo_vals   = [int(v) for v in emo_dist.values.tolist()]

# --- GKG : Top persons ---
all_persons = [p for p in ';'.join(df['GKG_Persons']).split(';') if p]
person_counter = Counter(all_persons).most_common(8)
person_labels = [p[0] for p in person_counter]
person_counts = [p[1] for p in person_counter]

# --- GKG : Top organizations ---
all_orgs = [o for o in ';'.join(df['GKG_Organizations']).split(';') if o]
org_counter = Counter(all_orgs).most_common(8)
org_labels = [o[0] for o in org_counter]
org_counts = [o[1] for o in org_counter]

# --- GKG : Theme categories by month ---
theme_cats_monthly = df.groupby('month')[['GKG_ThemeConflict','GKG_ThemeEconomy',
    'GKG_ThemeGovern','GKG_ThemeHumanDev','GKG_ThemeEnviro']].mean().tail(12)
tc_conflict = [round(float(v)*100, 1) for v in theme_cats_monthly['GKG_ThemeConflict'].tolist()]
tc_economy  = [round(float(v)*100, 1) for v in theme_cats_monthly['GKG_ThemeEconomy'].tolist()]
tc_govern   = [round(float(v)*100, 1) for v in theme_cats_monthly['GKG_ThemeGovern'].tolist()]
tc_human    = [round(float(v)*100, 1) for v in theme_cats_monthly['GKG_ThemeHumanDev'].tolist()]
tc_enviro   = [round(float(v)*100, 1) for v in theme_cats_monthly['GKG_ThemeEnviro'].tolist()]

sample = df[['date','Actor1Name','EventLabel','ActionGeo_FullName',
             'GoldsteinScale','AvgTone','ToneCategory','QuadLabel',
             'GKG_Themes','GCAM_DominantEmotion']].head(10)
sample_rows = []
for _, r in sample.iterrows():
    themes_short = r['GKG_Themes'].split(';')[0] if r['GKG_Themes'] else ''
    sample_rows.append({
        'date': str(r['date'])[:10],
        'actor': str(r['Actor1Name']),
        'event': str(r['EventLabel']),
        'city': str(r['ActionGeo_FullName']),
        'goldstein': round(float(r['GoldsteinScale']), 2),
        'tone': round(float(r['AvgTone']), 2),
        'sentiment': str(r['ToneCategory']),
        'quad': str(r['QuadLabel']),
        'theme': themes_short,
        'emotion': str(r['GCAM_DominantEmotion']),
    })

total_events  = len(df)
n_features    = int(df.shape[1])
avg_tone_val  = round(float(df['AvgTone'].mean()), 2)
conflict_pct  = round(float(df['IsConflict'].mean() * 100), 1)
goldstein_val = round(float(df['GoldsteinScale'].mean()), 2)
n_cities      = int(df['ActionGeo_FullName'].nunique())
period_start  = str(df['date'].min().date())
period_end    = str(df['date'].max().date())

tone_color    = '#D85A30' if avg_tone_val < 0 else '#1D9E75'
gold_color    = '#D85A30' if goldstein_val < 0 else '#1D9E75'
tone_sign     = '+' if avg_tone_val >= 0 else ''
gold_sign     = '+' if goldstein_val >= 0 else ''

n_themes_uniq = len(set(all_themes))
n_persons     = len(set(all_persons))
n_orgs        = len(set([o for o in all_orgs if o]))

print('Donnees dashboard preparees (Events + GKG)')
print(f'  {len(months_labels)} mois | {len(city_labels)} villes | {len(sample_rows)} lignes tableau')
print(f'  GKG: {n_themes_uniq} themes | {n_persons} personnes | {n_orgs} organisations | 6 GCAM emotions')


Donnees dashboard preparees
  12 mois | 8 villes | 10 lignes tableau


In [ ]:
HTML_TEMPLATE = '''
<!DOCTYPE html><html lang="fr"><head><meta charset="UTF-8">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{box-sizing:border-box;margin:0;padding:0;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}
body{background:#f8f7f4;color:#1a1a18;padding:16px;font-size:14px}
.header{background:#fff;border-radius:12px;padding:18px 22px;margin-bottom:14px;border:1px solid #e8e6de}
.h-title{font-size:19px;font-weight:600;margin-bottom:4px}
.h-sub{font-size:12px;color:#888;margin-bottom:10px}
.badge{display:inline-block;font-size:11px;padding:3px 10px;border-radius:20px;background:#E1F5EE;color:#0F6E56;font-weight:500;margin-right:6px;margin-bottom:4px}
.badge-gkg{background:#EDE7F6;color:#5E35B1}
.pipeline{display:flex;align-items:center;gap:5px;flex-wrap:wrap;margin-top:10px}
.ps{font-size:11px;padding:3px 10px;border-radius:14px;border:1px solid #e0ddd5;color:#5F5E5A;background:#fff}
.pa{font-size:11px;color:#B4B2A9}
.status{display:flex;align-items:center;gap:8px;font-size:12px;color:#5F5E5A;background:#f0f0ec;border-radius:8px;padding:8px 14px;margin-bottom:14px}
.sdot{width:8px;height:8px;border-radius:50%;background:#1D9E75;flex-shrink:0}
.metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-bottom:14px}
.metric{background:#fff;border-radius:10px;padding:14px 16px;border:1px solid #e8e6de}
.ml{font-size:11px;color:#888;margin-bottom:5px;font-weight:500;text-transform:uppercase;letter-spacing:.4px}
.mv{font-size:22px;font-weight:600}
.ms{font-size:11px;color:#B4B2A9;margin-top:2px}
.section-title{font-size:15px;font-weight:600;margin:18px 0 10px;padding-left:4px;display:flex;align-items:center;gap:8px}
.stag{font-size:10px;padding:2px 8px;border-radius:12px;font-weight:500}
.st-events{background:#E1F5EE;color:#0F6E56}
.st-gkg{background:#EDE7F6;color:#5E35B1}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px}
.card{background:#fff;border-radius:10px;padding:16px;border:1px solid #e8e6de}
.card.full{grid-column:1/-1}
.card-gkg{border-left:3px solid #7E57C2}
.ct{font-size:13px;font-weight:600;margin-bottom:2px}
.cs{font-size:11px;color:#888;margin-bottom:12px}
.legend{display:flex;flex-wrap:wrap;gap:10px;margin-bottom:10px;font-size:11px;color:#5F5E5A}
.li{display:flex;align-items:center;gap:4px}
.ld{width:10px;height:10px;border-radius:2px;display:inline-block;flex-shrink:0}
.bar-row{display:flex;align-items:center;gap:8px;margin-bottom:7px}
.bl{font-size:11px;color:#5F5E5A;width:100px;text-align:right;flex-shrink:0}
.bt{flex:1;height:16px;background:#f0f0ec;border-radius:4px;overflow:hidden}
.bf{height:100%;border-radius:4px}
.bv{font-size:11px;color:#888;width:50px;flex-shrink:0}
table{width:100%;border-collapse:collapse;font-size:12px}
th{text-align:left;color:#888;font-weight:500;padding:7px 8px;border-bottom:1px solid #f0f0ec;font-size:11px}
td{padding:7px 8px;border-bottom:1px solid #f8f7f4;color:#1a1a18}
tr:last-child td{border-bottom:none}
tr:hover td{background:#fafaf8}
.pill{display:inline-block;font-size:10px;padding:2px 8px;border-radius:12px;font-weight:500}
.pc{background:#E1F5EE;color:#0F6E56}
.pf{background:#FAECE7;color:#993C1D}
.pn{background:#FCEBEB;color:#A32D2D}
.pp{background:#EAF3DE;color:#3B6D11}
.pu{background:#F1EFE8;color:#5F5E5A}
.pg{background:#EDE7F6;color:#5E35B1}
</style></head><body>
CONTENT_PLACEHOLDER
</body></html>
'''

content = f'''
<div class="header">
  <div class="h-title">Benin Insights 2025 - Dashboard Pipeline</div>
  <div class="h-sub">BigQuery GDELT v2 (Events + GKG) - Data Engineer - iSHEERO x DataCamp Donates</div>
  <div>
    <span class="badge">GDELT Events</span>
    <span class="badge badge-gkg">GDELT GKG</span>
    <span class="badge">Benin (BN)</span>
    <span class="badge">2025-01-01 / 2025-12-31</span>
  </div>
  <div class="pipeline">
    <span class="ps">BigQuery Events</span><span class="pa">+</span>
    <span class="ps" style="border-color:#B39DDB">BigQuery GKG</span><span class="pa">-&gt;</span>
    <span class="ps">JOIN + Extraction</span><span class="pa">-&gt;</span>
    <span class="ps">Nettoyage</span><span class="pa">-&gt;</span>
    <span class="ps">{n_features} features</span><span class="pa">-&gt;</span>
    <span class="ps">gdelt_benin_clean.csv</span>
  </div>
</div>

<div class="status">
  <span class="sdot"></span>
  <span>Mode prototype - {total_events:,} evt - {n_cities} villes - {n_features} features - {period_start} a {period_end} - GKG: {n_themes_uniq} themes, {n_persons} pers, 6 GCAM</span>
</div>

<div class="metrics">
  <div class="metric"><div class="ml">Evenements total</div><div class="mv">{total_events:,}</div><div class="ms">Annee 2025 Benin</div></div>
  <div class="metric"><div class="ml">Features</div><div class="mv">{n_features}</div><div class="ms">Events + GKG combinees</div></div>
  <div class="metric"><div class="ml">AvgTone moyen</div><div class="mv" style="color:{tone_color}">{tone_sign}{avg_tone_val}</div><div class="ms">sentiment presse</div></div>
  <div class="metric"><div class="ml">Taux conflit</div><div class="mv" style="color:#D85A30">{conflict_pct}%</div><div class="ms">QuadClass &gt;= 3</div></div>
  <div class="metric"><div class="ml">Goldstein moyen</div><div class="mv" style="color:{gold_color}">{gold_sign}{goldstein_val}</div><div class="ms">stabilite politique</div></div>
  <div class="metric"><div class="ml">GKG Themes</div><div class="mv" style="color:#5E35B1">{n_themes_uniq}</div><div class="ms">themes uniques detectes</div></div>
</div>

<div class="section-title"><span>GDELT Events</span><span class="stag st-events">gdeltv2.events</span></div>

<div class="grid2">
  <div class="card">
    <div class="ct">Evenements par mois</div>
    <div class="cs">Volume mensuel GDELT - 2025</div>
    <div style="position:relative;height:200px"><canvas id="cMois"></canvas></div>
  </div>
  <div class="card">
    <div class="ct">QuadClass - Repartition</div>
    <div class="cs">Type evenements CAMEO</div>
    <div class="legend" id="qLeg"></div>
    <div style="position:relative;height:165px"><canvas id="cQuad"></canvas></div>
  </div>
  <div class="card">
    <div class="ct">AvgTone et GoldsteinScale</div>
    <div class="cs">Tonalite et stabilite par mois</div>
    <div class="legend">
      <span class="li"><span class="ld" style="background:#1D9E75"></span>AvgTone</span>
      <span class="li"><span class="ld" style="background:#D85A30"></span>Goldstein</span>
    </div>
    <div style="position:relative;height:180px"><canvas id="cTone"></canvas></div>
  </div>
  <div class="card">
    <div class="ct">Taux de conflit mensuel</div>
    <div class="cs">Pourcentage QuadClass &gt;= 3</div>
    <div style="position:relative;height:200px"><canvas id="cConflict"></canvas></div>
  </div>
  <div class="card">
    <div class="ct">Top 8 villes</div>
    <div class="cs">Volume par localite</div>
    <div id="barC"></div>
  </div>
  <div class="card">
    <div class="ct">Sentiment - Distribution</div>
    <div class="cs">Categories ToneCategory</div>
    <div style="position:relative;height:230px"><canvas id="cSent"></canvas></div>
  </div>
</div>

<div class="section-title"><span>GDELT GKG (Global Knowledge Graph)</span><span class="stag st-gkg">gdeltv2.gkg</span></div>

<div class="grid2">
  <div class="card card-gkg">
    <div class="ct">Top 10 Themes GKG</div>
    <div class="cs">V2Themes - Sujets les plus couverts</div>
    <div style="position:relative;height:250px"><canvas id="cThemes"></canvas></div>
  </div>
  <div class="card card-gkg">
    <div class="ct">GCAM - Emotions moyennes</div>
    <div class="cs">Global Content Analysis Measures - 6 dimensions</div>
    <div style="position:relative;height:250px"><canvas id="cGCAM"></canvas></div>
  </div>
  <div class="card card-gkg">
    <div class="ct">Top Personnes (V2Persons)</div>
    <div class="cs">Personnalites les plus mentionnees</div>
    <div id="barPersons"></div>
  </div>
  <div class="card card-gkg">
    <div class="ct">Top Organisations (V2Organizations)</div>
    <div class="cs">Institutions et entites les plus citees</div>
    <div id="barOrgs"></div>
  </div>
  <div class="card card-gkg">
    <div class="ct">Emotion dominante - Distribution</div>
    <div class="cs">GCAM_DominantEmotion par evenement</div>
    <div style="position:relative;height:220px"><canvas id="cEmoDist"></canvas></div>
  </div>
  <div class="card card-gkg">
    <div class="ct">Themes par mois (%)</div>
    <div class="cs">Evolution des categories thematiques GKG</div>
    <div class="legend">
      <span class="li"><span class="ld" style="background:#D85A30"></span>Conflit/Securite</span>
      <span class="li"><span class="ld" style="background:#378ADD"></span>Economie</span>
      <span class="li"><span class="ld" style="background:#1D9E75"></span>Gouvernance</span>
      <span class="li"><span class="ld" style="background:#EF9F27"></span>Dvlpt humain</span>
      <span class="li"><span class="ld" style="background:#7E57C2"></span>Environnement</span>
    </div>
    <div style="position:relative;height:200px"><canvas id="cThemeMonth"></canvas></div>
  </div>
</div>

<div class="grid2">
  <div class="card full">
    <div class="ct">Echantillon dataset - gdelt_benin_clean.csv (Events + GKG)</div>
    <div class="cs">10 premiers evenements avec feature engineering</div>
    <div style="overflow-x:auto">
      <table><thead><tr>
        <th>Date</th><th>Acteur</th><th>Evenement</th><th>Ville</th>
        <th>Goldstein</th><th>AvgTone</th><th>Sentiment</th><th>QuadClass</th>
        <th>Theme GKG</th><th>Emotion</th>
      </tr></thead><tbody id="tBody"></tbody></table>
    </div>
  </div>
</div>

<script>
var months   = ''' + json.dumps(months_labels) + ''';
var evD      = ''' + json.dumps(months_vals) + ''';
var toneD    = ''' + json.dumps(tone_vals) + ''';
var goldD    = ''' + json.dumps(gold_vals) + ''';
var conflD   = ''' + json.dumps(conflict_vals) + ''';
var qLabels  = ''' + json.dumps(quad_labels_list) + ''';
var qVals    = ''' + json.dumps(quad_vals_list) + ''';
var cLabels  = ''' + json.dumps(city_labels) + ''';
var cVals    = ''' + json.dumps(city_counts) + ''';
var sLabels  = ''' + json.dumps(tone_cats) + ''';
var sVals    = ''' + json.dumps(tone_cnts) + ''';
var rows     = ''' + json.dumps(sample_rows) + ''';
var thLabels = ''' + json.dumps(theme_labels) + ''';
var thVals   = ''' + json.dumps(theme_counts) + ''';
var gcamLbls = ''' + json.dumps(gcam_labels) + ''';
var gcamVals = ''' + json.dumps(gcam_means) + ''';
var emoLbls  = ''' + json.dumps(emo_labels) + ''';
var emoVals  = ''' + json.dumps(emo_vals) + ''';
var pLabels  = ''' + json.dumps(person_labels) + ''';
var pVals    = ''' + json.dumps(person_counts) + ''';
var oLabels  = ''' + json.dumps(org_labels) + ''';
var oVals    = ''' + json.dumps(org_counts) + ''';
var tcConf   = ''' + json.dumps(tc_conflict) + ''';
var tcEco    = ''' + json.dumps(tc_economy) + ''';
var tcGov    = ''' + json.dumps(tc_govern) + ''';
var tcHum    = ''' + json.dumps(tc_human) + ''';
var tcEnv    = ''' + json.dumps(tc_enviro) + ''';

var QC=["#1D9E75","#378ADD","#EF9F27","#D85A30"];
var SC=["#E24B4A","#D85A30","#888780","#1D9E75","#3B6D11"];
var GKG_C=["#7E57C2","#5E35B1","#AB47BC","#BA68C8","#9575CD","#CE93D8","#B39DDB","#E1BEE7","#D1C4E9","#F3E5F5"];
var EMO_C=["#E24B4A","#EF9F27","#1D9E75","#378ADD","#7E57C2","#BA68C8"];

new Chart(document.getElementById("cMois"),{type:"bar",data:{labels:months,datasets:[{label:"Evenements",data:evD,backgroundColor:"rgba(29,158,117,0.75)",borderRadius:4}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{x:{ticks:{font:{size:10},autoSkip:false,maxRotation:45}},y:{ticks:{font:{size:10}}}}}});

var ql=document.getElementById("qLeg");
qLabels.forEach(function(l,i){var pct=Math.round(qVals[i]/qVals.reduce(function(a,b){return a+b;},0)*100);ql.innerHTML+='<span class="li"><span class="ld" style="background:'+QC[i]+'"></span>'+l+' '+pct+'%</span>';});
new Chart(document.getElementById("cQuad"),{type:"doughnut",data:{labels:qLabels,datasets:[{data:qVals,backgroundColor:QC,borderWidth:0}]},options:{responsive:true,maintainAspectRatio:false,cutout:"60%",plugins:{legend:{display:false}}}});

new Chart(document.getElementById("cTone"),{type:"line",data:{labels:months,datasets:[{label:"AvgTone",data:toneD,borderColor:"#1D9E75",backgroundColor:"rgba(29,158,117,0.08)",tension:0.4,pointRadius:3,fill:true},{label:"Goldstein",data:goldD,borderColor:"#D85A30",backgroundColor:"rgba(216,90,48,0.05)",tension:0.4,pointRadius:3,borderDash:[4,3]}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{x:{ticks:{font:{size:10},autoSkip:false,maxRotation:45}},y:{ticks:{font:{size:10}}}}}});

new Chart(document.getElementById("cConflict"),{type:"line",data:{labels:months,datasets:[{label:"Conflit %",data:conflD,borderColor:"#D85A30",backgroundColor:"rgba(216,90,48,0.1)",tension:0.4,fill:true,pointRadius:4}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{x:{ticks:{font:{size:10},autoSkip:false,maxRotation:45}},y:{min:0,max:60,ticks:{font:{size:10},callback:function(v){return v+"%";}}}}}});

var maxC=Math.max.apply(null,cVals);var bc=document.getElementById("barC");var bColors=["#1D9E75","#1D9E75","#378ADD","#378ADD","#888780","#888780","#888780","#888780"];
bc.innerHTML=cLabels.map(function(c,i){return '<div class="bar-row"><div class="bl">'+c+'</div><div class="bt"><div class="bf" style="width:'+Math.round(cVals[i]/maxC*100)+'%;background:'+bColors[i]+'"></div></div><div class="bv">'+cVals[i].toLocaleString()+'</div></div>';}).join("");

new Chart(document.getElementById("cSent"),{type:"bar",data:{labels:sLabels,datasets:[{label:"Evenements",data:sVals,backgroundColor:SC,borderRadius:4}]},options:{responsive:true,maintainAspectRatio:false,indexAxis:"y",plugins:{legend:{display:false}},scales:{x:{ticks:{font:{size:10}}},y:{ticks:{font:{size:10}}}}}});

/* GKG Charts */
new Chart(document.getElementById("cThemes"),{type:"bar",data:{labels:thLabels,datasets:[{label:"Mentions",data:thVals,backgroundColor:GKG_C,borderRadius:4}]},options:{responsive:true,maintainAspectRatio:false,indexAxis:"y",plugins:{legend:{display:false}},scales:{x:{ticks:{font:{size:10}}},y:{ticks:{font:{size:9}}}}}});

new Chart(document.getElementById("cGCAM"),{type:"radar",data:{labels:gcamLbls,datasets:[{label:"Score moyen",data:gcamVals,backgroundColor:"rgba(126,87,194,0.15)",borderColor:"#7E57C2",pointBackgroundColor:"#5E35B1",pointRadius:4}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{r:{beginAtZero:true,ticks:{font:{size:9}},pointLabels:{font:{size:11}}}}}});

var maxP=Math.max.apply(null,pVals);var bp=document.getElementById("barPersons");
bp.innerHTML=pLabels.map(function(p,i){return '<div class="bar-row"><div class="bl" style="width:140px">'+p+'</div><div class="bt"><div class="bf" style="width:'+Math.round(pVals[i]/maxP*100)+'%;background:#7E57C2"></div></div><div class="bv">'+pVals[i]+'</div></div>';}).join("");

var maxO=Math.max.apply(null,oVals);var bo=document.getElementById("barOrgs");
bo.innerHTML=oLabels.map(function(o,i){return '<div class="bar-row"><div class="bl" style="width:160px">'+o+'</div><div class="bt"><div class="bf" style="width:'+Math.round(oVals[i]/maxO*100)+'%;background:#AB47BC"></div></div><div class="bv">'+oVals[i]+'</div></div>';}).join("");

new Chart(document.getElementById("cEmoDist"),{type:"doughnut",data:{labels:emoLbls,datasets:[{data:emoVals,backgroundColor:EMO_C,borderWidth:0}]},options:{responsive:true,maintainAspectRatio:false,cutout:"55%",plugins:{legend:{position:"right",labels:{font:{size:10}}}}}});

new Chart(document.getElementById("cThemeMonth"),{type:"line",data:{labels:months,datasets:[{label:"Conflit",data:tcConf,borderColor:"#D85A30",tension:0.4,pointRadius:2,borderWidth:2},{label:"Economie",data:tcEco,borderColor:"#378ADD",tension:0.4,pointRadius:2,borderWidth:2},{label:"Gouvernance",data:tcGov,borderColor:"#1D9E75",tension:0.4,pointRadius:2,borderWidth:2},{label:"Dvlpt humain",data:tcHum,borderColor:"#EF9F27",tension:0.4,pointRadius:2,borderWidth:2},{label:"Environnement",data:tcEnv,borderColor:"#7E57C2",tension:0.4,pointRadius:2,borderWidth:2,borderDash:[4,3]}]},options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{x:{ticks:{font:{size:10},autoSkip:false,maxRotation:45}},y:{ticks:{font:{size:10},callback:function(v){return v+"%";}}}}}});

document.getElementById("tBody").innerHTML=rows.map(function(r){var gc=r.goldstein>=0?"#1D9E75":"#D85A30";var tc=r.tone>=0?"#1D9E75":"#D85A30";var qc=r.quad.indexOf("Conflit")>=0?"pf":"pc";var sc=(r.sentiment.indexOf("negatif")>=0||r.sentiment.indexOf("Negatif")>=0)?"pn":(r.sentiment.indexOf("Positif")>=0?"pp":"pu");var gs=(r.goldstein>=0?"+":"")+r.goldstein.toFixed(1);var ts=(r.tone>=0?"+":"")+r.tone.toFixed(1);return "<tr><td>"+r.date+"</td><td>"+r.actor+"</td><td>"+r.event+"</td><td>"+r.city+"</td><td style=\"font-weight:600;color:"+gc+"\">"+gs+"</td><td style=\"font-weight:600;color:"+tc+"\">"+ts+"</td><td><span class=\"pill "+sc+"\">"+r.sentiment+"</span></td><td><span class=\"pill "+qc+"\">"+r.quad+"</span></td><td><span class=\"pill pg\">"+r.theme+"</span></td><td>"+r.emotion+"</td></tr>";}).join("");
</script>
'''

dashboard_html = HTML_TEMPLATE.replace('CONTENT_PLACEHOLDER', content)
display(HTML(dashboard_html))
print('Dashboard Events + GKG affiche dans Colab')


Date,Acteur,Evenement,Ville,Goldstein,AvgTone,Sentiment,QuadClass


Dashboard affiche dans Colab


## 4 - Sauvegarder et telecharger les fichiers

In [ ]:
import os, json
from pathlib import Path

os.makedirs('data/processed', exist_ok=True)
os.makedirs('dashboard', exist_ok=True)

csv_path = 'data/processed/gdelt_benin_clean.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'CSV : {csv_path} ({Path(csv_path).stat().st_size/1e6:.1f} MB)')

html_path = 'dashboard/index.html'
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(dashboard_html)
print(f'HTML: {html_path} ({Path(html_path).stat().st_size/1e3:.0f} KB)')

# Compter themes, personnes, orgs pour metadata
from collections import Counter
all_t = ';'.join(df['GKG_Themes']).split(';')
all_p = [p for p in ';'.join(df['GKG_Persons']).split(';') if p]
all_o = [o for o in ';'.join(df['GKG_Organizations']).split(';') if o]

meta = {
    'generated_at'    : datetime.now().isoformat(),
    'source'          : 'GDELT v2 prototype (Events + GKG)',
    'tables'          : ['gdeltv2.events', 'gdeltv2.gkg'],
    'country'         : 'Benin (BN)',
    'period_start'    : '2025-01-01',
    'period_end'      : '2025-12-31',
    'rows'            : len(df),
    'columns'         : df.shape[1],
    'avg_tone'        : round(float(df['AvgTone'].mean()), 4),
    'goldstein'       : round(float(df['GoldsteinScale'].mean()), 4),
    'conflict_pct'    : round(float(df['IsConflict'].mean()*100), 2),
    'gkg_themes_unique'      : len(set(all_t)),
    'gkg_persons_unique'     : len(set(all_p)),
    'gkg_organizations_unique': len(set(all_o)),
    'gcam_emotions'          : 6,
    'gcam_dominant_emotion'  : df['GCAM_DominantEmotion'].value_counts().index[0],
    'features'               : list(df.columns),
}
meta_path = 'data/processed/dataset_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f'Meta: {meta_path}')

try:
    from google.colab import files
    print('\nTelechargement...')
    files.download(csv_path)
    files.download(meta_path)
    files.download(html_path)
    print('3 fichiers telecharges')
except ImportError:
    print('\nFichiers sauvegardes localement.')
    print('Ouvrir dashboard/index.html dans le navigateur')


CSV : data/processed/gdelt_benin_clean.csv (1.7 MB)
HTML: dashboard/index.html (14 KB)
Meta: data/processed/dataset_metadata.json

Telechargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

3 fichiers telecharges
